# Project- Music Recommendation System

In [ ]:
!pip install -q langchain-community langchain-huggingface langchain-chroma
!pip install -q gradio

In [ ]:
import json
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_chroma import Chroma
from google.colab import files
import gradio as gr

In [ ]:
uploaded = files.upload()
json_filename = list(uploaded.keys())[0]

In [ ]:
with open(json_filename, "r", encoding="utf-8") as f:
    songs = json.load(f)

print(f"Loaded {len(songs)} songs.")

In [ ]:
TITLE_KEY = "Song Name"
ARTIST_KEY = "Artist"
GENRE_KEY = "Genre"
MOOD_KEY = "Mood"
DESC_KEY = "Description"
LANG_KEY = "Language"
YEAR_KEY = "Release Year"

In [ ]:
docs = []
for song in songs:
    text = f"{song[GENRE_KEY]} {song[MOOD_KEY]} {song[ARTIST_KEY]} {song[DESC_KEY]}"
    docs.append(Document(page_content=text, metadata={
        "title": song[TITLE_KEY],
        "artist": song[ARTIST_KEY],
        "genre": song[GENRE_KEY],
        "mood": song[MOOD_KEY],
        "language": song[LANG_KEY],
        "year": song[YEAR_KEY]
    }))

In [ ]:
encoder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [ ]:
vector_db = Chroma.from_documents(
    documents=docs,
    embedding=encoder,
    persist_directory="song_db_v2",
    collection_name="songVectorDB_v2",
    collection_metadata={"hnsw:space": "cosine"},
)

In [ ]:
def get_recommendations(song_title, query_text, genre, mood, language, min_year, top_n=5):
    if query_text and query_text.strip():
        search_text = query_text.strip()
    elif song_title:
        song = next((s for s in songs if s[TITLE_KEY] == song_title), None)
        if song is None:
            return "Song not found."
        search_text = f"{song[GENRE_KEY]} {song[MOOD_KEY]} {song[ARTIST_KEY]} {song[DESC_KEY]}"
    else:
        return "Please select a song or type a search query."

    results = vector_db.similarity_search_with_score(query=search_text, k=len(songs))

    recs = []
    for doc, score in results:
        meta = doc.metadata

        if song_title and meta["title"] == song_title:
            continue
        if genre and genre != "All" and meta["genre"] != genre:
            continue
        if mood and mood != "All" and meta["mood"] != mood:
            continue
        if language and language != "All" and meta["language"] != language:
            continue
        if meta["year"] < min_year:
            continue

        recs.append({
            "title": meta["title"],
            "artist": meta["artist"],
            "genre": meta["genre"],
            "mood": meta["mood"],
            "score": round(float(score), 4)
        })
        if len(recs) == top_n:
            break

    return recs if recs else "No songs matched your filters."

In [ ]:
def show_recommendations(song_title, query_text, genre, mood, language, min_year):

    recs = get_recommendations(song_title, query_text, genre, mood, language, min_year)

    if isinstance(recs, str):
        return f"<p style='color:#ef4444;font-weight:bold;font-family:sans-serif;'>{recs}</p>"

    heading = f"'{query_text}'" if query_text and query_text.strip() else f"'{song_title}'"
    html = f"<h3 style='font-family:sans-serif;color:#38bdf8;margin-top:20px;'>Top songs matching {heading}</h3>"

    for rec in recs:
        html += f"""
        <div style="border:1px solid #334155; border-radius:12px; padding:16px; margin-bottom:12px; background:#1e293b; box-shadow:0 4px 12px rgba(0,0,0,0.5);">
            <h4 style="margin:0 0 8px 0; color:#38bdf8; font-family:sans-serif; font-size: 1.1em;">🎵 {rec['title']}</h4>
            <p style="margin:4px 0; color:#cbd5e1; font-family:sans-serif;">🎤 <strong style="color:#f1f5f9;">Artist:</strong> {rec['artist']}</p>
            <p style="margin:4px 0; color:#cbd5e1; font-family:sans-serif;">🎼 <strong style="color:#f1f5f9;">Genre:</strong> {rec['genre']}</p>
            <p style="margin:4px 0; color:#cbd5e1; font-family:sans-serif;">😊 <strong style="color:#f1f5f9;">Mood:</strong> {rec['mood']}</p>
            <p style="margin:4px 0; color:#cbd5e1; font-family:sans-serif;">📊 <strong style="color:#f1f5f9;">Similarity Score:</strong>
                <code style="background:#0f172a; color:#38bdf8; padding:2px 8px; border-radius:6px; font-weight:bold; border: 1px solid #334155;">{rec['score']}</code>
            </p>
        </div>
        """
    return html

In [ ]:
custom_css = """
.gradio-container {
    background-color: #0f172a !important;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
    color: #f1f5f9 !important;
}
.gradio-container h1, .gradio-container h2, .gradio-container p, .gradio-container span {
    color: #f1f5f9 !important;
}
#results_box {
    background-color: transparent !important;
    border: none !important;
}
"""

genres = sorted(set(song[GENRE_KEY] for song in songs))
moods = sorted(set(song[MOOD_KEY] for song in songs))
languages = sorted(set(song[LANG_KEY] for song in songs))
years = sorted(set(song[YEAR_KEY] for song in songs))

with gr.Blocks() as demo:
    gr.Markdown("# 🎧 Music Recommendation System")
    gr.Markdown("Select a song OR type a request (e.g. *'relaxing songs for studying'*), then narrow down with filters.")

    song_dropdown = gr.Dropdown(
        choices=sorted([song[TITLE_KEY] for song in songs]),
        value=None,
        label="Select a Song",
        info="Choose a song from the list...",
        interactive=True
    )

    query_box = gr.Textbox(
        label="Or type a request",
        placeholder="e.g. suggest romantic songs, recommend workout songs..."
    )

    with gr.Row():
        genre_filter = gr.Dropdown(choices=["All"] + genres, value="All", label="🎼 Genre")
        mood_filter = gr.Dropdown(choices=["All"] + moods, value="All", label="😊 Mood")
        language_filter = gr.Dropdown(choices=["All"] + languages, value="All", label="🌍 Language")

    year_slider = gr.Slider(
        minimum=min(years), maximum=max(years), value=min(years), step=1,
        label="📅 Released after year"
    )

    recommend_button = gr.Button("Get Recommendations", variant="primary")
    results_output = gr.HTML(elem_id="results_box")

    inputs = [song_dropdown, query_box, genre_filter, mood_filter, language_filter, year_slider]
    recommend_button.click(fn=show_recommendations, inputs=inputs, outputs=results_output)

demo.launch(
    share=True,
    theme=gr.themes.Soft(),
    css=custom_css,
    js="() => { document.querySelector('body').classList.add('dark'); }"
)